# Brian2 Spike Cascade: Touch Stimulus Through Real Wiring

Stimulate the C. elegans touch sensory neurons and watch the spike cascade propagate through **real biological wiring** to interneurons and motor neurons. This is the same principle behind the Eon Systems fruit fly brain — behavior emerging from connectome architecture.

In [ ]:
import sys
sys.path.insert(0, "../creatures-core")

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import TwoSlopeNorm

from creatures.connectome.openworm import load
from creatures.connectome.types import NeuronType
from creatures.neural.brian2_engine import Brian2Engine
from creatures.neural.base import NeuralConfig

%matplotlib inline
plt.rcParams["figure.figsize"] = (14, 8)
plt.rcParams["figure.dpi"] = 100

In [ ]:
# Load connectome and build neural network
connectome = load("edge_list")

config = NeuralConfig(
    weight_scale=3.0,   # mV per unit synapse weight
    tau_syn=8.0,        # synaptic current decay (ms)
    tau_m=15.0,         # membrane time constant (ms)
)

engine = Brian2Engine()
engine.build(connectome, config)
print(f"Network: {engine.n_neurons} neurons")

## Simulate Touch Stimulus

Inject current into all 5 gentle-touch sensory neurons (ALML, ALMR, AVM, PLML, PLMR) — simulating a body touch. Run for 200ms and record all spikes.

In [ ]:
# Stimulate touch sensory neurons
engine.set_input_currents({
    "ALML": 25.0, "ALMR": 25.0, "AVM": 25.0,
    "PLML": 25.0, "PLMR": 25.0,
})

# Run simulation
for _ in range(200):
    engine.step(1.0)

# Get spike history
spike_indices, spike_times = engine.get_spike_history()
print(f"Total spikes: {len(spike_indices)}")
print(f"Active neurons: {len(set(spike_indices))} / {engine.n_neurons}")

## Raster Plot

Each dot is a spike. Neurons are grouped by type (sensory → interneuron → motor). Watch the cascade: sensory neurons fire first, then the signal propagates through interneurons to motor neurons.

In [ ]:
# Build sorted neuron ordering (sensory → inter → motor) with only active neurons
type_order = {NeuronType.SENSORY: 0, NeuronType.INTER: 1, NeuronType.MOTOR: 2}
type_colors = {NeuronType.SENSORY: "#4CAF50", NeuronType.INTER: "#2196F3", NeuronType.MOTOR: "#FF5722"}
type_labels = {NeuronType.SENSORY: "Sensory", NeuronType.INTER: "Interneuron", NeuronType.MOTOR: "Motor"}

# Only include neurons that actually fired
active_neurons = sorted(set(spike_indices))
active_ids = [engine.neuron_ids[i] for i in active_neurons]
sorted_active = sorted(active_ids,
    key=lambda nid: (type_order.get(connectome.neurons[nid].neuron_type, 3), nid))

# Create mapping from original index to raster y-position
raster_map = {engine.neuron_ids.index(nid): y for y, nid in enumerate(sorted_active)}

fig, ax = plt.subplots(figsize=(16, 10))

# Plot spikes colored by neuron type
for idx, t in zip(spike_indices, spike_times):
    if idx in raster_map:
        nid = engine.neuron_ids[idx]
        ntype = connectome.neurons[nid].neuron_type
        color = type_colors.get(ntype, "#999")
        ax.plot(t, raster_map[idx], "|", color=color, markersize=3, alpha=0.7)

# Draw type boundaries and labels
boundaries = []
current_type = connectome.neurons[sorted_active[0]].neuron_type
for y, nid in enumerate(sorted_active):
    if connectome.neurons[nid].neuron_type != current_type:
        boundaries.append(y)
        current_type = connectome.neurons[nid].neuron_type

for b in boundaries:
    ax.axhline(b - 0.5, color="gray", linewidth=0.5, linestyle="--", alpha=0.5)

# Type region labels
starts = [0] + boundaries
ends = boundaries + [len(sorted_active)]
types_in_order = [NeuronType.SENSORY, NeuronType.INTER, NeuronType.MOTOR]
for ntype, start, end in zip(types_in_order, starts, ends):
    if end > start:
        mid = (start + end) / 2
        ax.text(-5, mid, type_labels[ntype], ha="right", va="center",
                fontsize=11, fontweight="bold", color=type_colors[ntype])

ax.set_xlim(0, 200)
ax.set_ylim(-1, len(sorted_active))
ax.set_xlabel("Time (ms)", fontsize=13)
ax.set_ylabel("Neuron (sorted by type)", fontsize=13)
ax.set_title("Spike Raster: Touch Stimulus → Neural Cascade Through Real C. elegans Wiring",
             fontsize=14, fontweight="bold")
ax.set_yticks([])
plt.tight_layout()
plt.show()

## Voltage Traces: Key Neurons in the Touch Pathway

Follow the signal from sensory input (PLML) → command interneuron (AVAL) → motor neuron (VA1).

In [ ]:
# Voltage traces for key neurons in the touch withdrawal pathway
key_neurons = [
    ("PLML", "Touch sensor (posterior)", NeuronType.SENSORY),
    ("ALML", "Touch sensor (anterior)", NeuronType.SENSORY),
    ("AVAL", "Command interneuron (backward)", NeuronType.INTER),
    ("AVBL", "Command interneuron (forward)", NeuronType.INTER),
    ("PVCL", "Command interneuron", NeuronType.INTER),
    ("VA1", "Ventral motor (backward)", NeuronType.MOTOR),
    ("DA1", "Dorsal motor (backward)", NeuronType.MOTOR),
    ("VB1", "Ventral motor (forward)", NeuronType.MOTOR),
]

traces = engine.get_voltage_history([nid for nid, _, _ in key_neurons])

fig, axes = plt.subplots(len(key_neurons), 1, figsize=(16, 14), sharex=True)

for ax, (nid, label, ntype) in zip(axes, key_neurons):
    if nid in traces:
        times, voltages = traces[nid]
        color = type_colors[ntype]
        ax.plot(times, voltages, color=color, linewidth=0.8, alpha=0.9)
        ax.axhline(-50, color="red", linewidth=0.5, linestyle=":", alpha=0.4)
        ax.set_ylabel("mV", fontsize=9)
        ax.set_ylim(-70, -30)
        ax.text(0.01, 0.95, f"{nid} — {label}", transform=ax.transAxes,
                fontsize=10, fontweight="bold", va="top", color=color)
        ax.tick_params(labelsize=8)

axes[-1].set_xlabel("Time (ms)", fontsize=12)
fig.suptitle("Voltage Traces: Touch Stimulus Propagation Through Real Neural Wiring",
             fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

## Motor Neuron Firing Rates

These are the firing rates that would drive muscle contraction. In the next step, we'll connect these to a simulated worm body.

In [ ]:
# Motor neuron firing rates
rates = engine.get_firing_rates()
motor_neurons = connectome.neurons_by_type(NeuronType.MOTOR)
motor_rates = {n.id: rates.get(n.id, 0) for n in motor_neurons}

# Separate into forward (VB, DB) and backward (VA, DA) motor pools
forward_ids = sorted([nid for nid in motor_rates if nid.startswith(("VB", "DB"))])
backward_ids = sorted([nid for nid in motor_rates if nid.startswith(("VA", "DA"))])
inhibitory_ids = sorted([nid for nid in motor_rates if nid.startswith(("VD", "DD"))])

fig, axes = plt.subplots(3, 1, figsize=(14, 8), sharex=True)

for ax, ids, label, color in [
    (axes[0], backward_ids, "Backward motor (VA, DA) — reversal", "#E91E63"),
    (axes[1], forward_ids, "Forward motor (VB, DB) — forward crawl", "#2196F3"),
    (axes[2], inhibitory_ids, "Inhibitory motor (VD, DD) — muscle relaxation", "#00BCD4"),
]:
    values = [motor_rates[nid] for nid in ids]
    ax.bar(range(len(ids)), values, color=color, alpha=0.8)
    ax.set_xticks(range(len(ids)))
    ax.set_xticklabels(ids, rotation=90, fontsize=7)
    ax.set_ylabel("Hz", fontsize=10)
    ax.set_title(label, fontsize=11, fontweight="bold", color=color)

fig.suptitle("Motor Neuron Firing Rates — Ready to Drive a Virtual Body",
             fontsize=14, fontweight="bold")
plt.tight_layout()
plt.show()

# Summary
total_backward = sum(motor_rates[n] for n in backward_ids)
total_forward = sum(motor_rates[n] for n in forward_ids)
print(f"Backward motor drive: {total_backward:.0f} Hz total")
print(f"Forward motor drive: {total_forward:.0f} Hz total")
print(f"Ratio (backward/forward): {total_backward/max(total_forward, 1):.1f}x")
print(f"\n→ The worm should move BACKWARD (touch withdrawal reflex)!")